In [ ]:
!pip install adjustText

In [ ]:
#Active l'affichage des graphiques directement dans le notebook
%matplotlib inline

import zipfile
import re
import numpy as np
import pandas as pd
import os
import random
import tensorflow as tf
import matplotlib.pyplot as plt
from six.moves.urllib.request import urlretrieve  #Pour télécharger des fichiers depuis une URL
from sklearn.manifold import TSNE #Pour la réduction de dimensions et visualisation
from adjustText import adjust_text #Pour ajuster les annotations de texte dans les graphes


# Fonction pour télécharger et extraire les données

In [ ]:
def download_data(url, data_dir):

    os.makedirs(data_dir, exist_ok=True)
    file_path = os.path.join(data_dir, 'bbcsport-fulltext.zip')

    if not os.path.exists(file_path):
        print('Télechargement des fichiers...')
        filename, _ = urlretrieve(url, file_path)
    else:
        print("Fichier déjà existe")

    extract_path = os.path.join(data_dir, 'bbc')

    #Vérifie si les fichiers sont déjà extraits, sinon les extrait
    if not os.path.exists(extract_path):

        with zipfile.ZipFile(
            os.path.join(data_dir, 'bbcsport-fulltext.zip'),
        'r'
        ) as zipf:
            zipf.extractall(data_dir)

    else:
        print("bbc-fulltext.zip est déjà extracté")



In [ ]:
url = 'http://mlg.ucd.ie/files/datasets/bbcsport-fulltext.zip'

download_data(url, 'Data_Folder')


Télechargement des fichiers...


Parcourt les fichiers texte dans le répertoire donné et extrait le contenu des articles.
Retourne une liste de chaînes de caractères, chaque chaîne représentant un article.

In [ ]:
def lire_articles(repertoire_donnees):

    articles = []  #Liste pour stocker les articles
    noms_fichiers = []  #Liste pour stocker les noms des fichiers
    print("Lecture des fichiers en cours...")

    #Parcours récursif des fichiers dans le dossier
    for racine, dossiers, fichiers in os.walk(repertoire_donnees):
        for index, nom_fichier in enumerate(fichiers):
            #On ignore les fichiers README
            if 'README' in nom_fichier:
                continue

            #Affichage progressif pour le suivi
            print("." * index, nom_fichier, end='\r')

            #Lecture du fichier
            chemin_complet = os.path.join(racine, nom_fichier)
            with open(chemin_complet, encoding='latin-1') as fichier:
                lignes = [ligne.strip() for ligne in fichier]  #Nettoyage des lignes (enlève les sauts de ligne)
                article = ' '.join(lignes)  #Regroupe les lignes en une seule chaîne
                articles.append(article)  #Ajoute l'article à la liste
                noms_fichiers.append(nom_fichier)

    print(f"\nNombre total d'articles détectés : {len(articles)}")
    return articles, noms_fichiers

In [ ]:
#Lecture des articles à partir du dossier "data/bbc"
articles , noms_fichiers = lire_articles(os.path.join('Data_Folder', 'bbcsport'))

#Calcul du nombre total de mots dans l'ensemble des articles
nombre_total_mots = sum(len(article.split(' ')) for article in articles)
print(f"{nombre_total_mots} mots trouvés dans l'ensemble des articles.")

print("Nom du fichier :", noms_fichiers[0])

#Affichage de quelques exemples de texte (début et fin des articles)
print('Exemple de début d’un article : ', articles[0][:50])
print('Exemple de fin d’un article : ', articles[-1][-50:])


Lecture des fichiers en cours...
........................................................................................................................................................................................................................................................................ 230.txt
Nombre total d'articles détectés : 737
254826 mots trouvés dans l'ensemble des articles.
Nom du fichier : 009.txt
Exemple de début d’un article :  Wales coach elated with win  Mike Ruddock paid tri
Exemple de fin d’un article :  d. "Chelsea are one of the highest-ranking teams."


# Création d'un objet Tokenizer pour transformer les textes en séquences de tokens (entiers)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokeniseur = Tokenizer(
    num_words=None,  #On garde tous les mots du corpus (pas de limite sur le vocabulaire)
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',  #Caractères à ignorer lors du traitement
    lower=True,      #Convertir tous les mots en minuscules
    split=' '        #séparation des mots selon l'espace
)


In [ ]:
tokeniseur.fit_on_texts(articles)

# Calcul de la taille du vocabulaire (nombre total de mots uniques + 1 pour le token spécial 0)

In [ ]:
taille_vocabulaire = len(tokeniseur.word_index.items()) + 1
print(f"Taille du vocabulaire : {taille_vocabulaire}")

#affichage des 10 premiers mots dans l'index (souvent les plus fréquents)
print("\nMots les plus fréquents :")
print('\t', dict(list(tokeniseur.word_index.items())[:10]))

#affichage des 10 derniers mots (souvent les moins fréquents)
print("\nMots les moins fréquents :")
print('\t', dict(list(tokeniseur.word_index.items())[-10:]))


Taille du vocabulaire : 14225

Mots les plus fréquents :
	 {'the': 1, 'to': 2, 'a': 3, 'and': 4, 'in': 5, 'of': 6, 'for': 7, 'he': 8, 'on': 9, 'i': 10}

Mots les moins fréquents :
	 {'wonderfully': 14215, 'clinically': 14216, 'hacked': 14217, 'teasing': 14218, 'dejan': 14219, "ashdown's": 14220, 'mezague': 14221, '267': 14222, 'gripped': 14223, 'reassure': 14224}


# Création du tokenizer avec une limite sur la taille du vocabulaire

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokeniseur = Tokenizer(
    num_words=15000,  #On ne garde que les 15 000 mots les plus fréquents
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
    lower=True,
    split=' ',
    oov_token=''      # Mot spécial pour gérer les mots inconnus (Out Of Vocabulary)
)

#Apprentissage du vocabulaire à partir de la liste des articles
tokeniseur.fit_on_texts(articles)


In [ ]:
#affichage de 100 premiers caractères du premier article
print(f"Texte original : {articles[0][:100]}")

#affichage de la séquence de tokens correspondant à ce texte
print(f"IDs des mots (séquence) : {tokeniseur.texts_to_sequences([articles[0][:100]])[0]}")


Texte original : Wales coach elated with win  Mike Ruddock paid tribute to his Wales side after they came from 15-6 d
IDs des mots (séquence) : [130, 108, 8848, 15, 54, 597, 858, 1127, 2196, 3, 18, 130, 81, 29, 33, 189, 27, 543, 71, 305]


# Conversion de tous les articles en listes de tokens (séquences d'IDs)

In [ ]:
sequences_articles = tokeniseur.texts_to_sequences(articles)

#Génération de skip-grams à partir du corpus

Dans TensorFlow, on dispose la fonction pratique tf.keras.preprocessing sequence.skipgrams() pour générer des skip-grams.










In [ ]:
#On récupère les 5 premiers IDs de mots de la première séquence d'article
ids_mots_exemple = sequences_articles[0][:5]

#Reconstruction de la phrase à partir des IDs, en utilisant l'index inverse du tokenizer
phrase_exemple = ' '.join([tokeniseur.index_word[wid] for wid in ids_mots_exemple])

#Affichage de la phrase reconstituée
print(f"Phrase exemple : {phrase_exemple}")

#Affichage des IDs des mots correspondants
print(f"IDs des mots exemple : {ids_mots_exemple}\n")

Phrase exemple : wales coach elated with win
IDs des mots exemple : [130, 108, 8848, 15, 54]



In [ ]:
#Taille de la fenêtre contextuelle autour du mot cible
taille_fenetre = 1

#Génération des paires (mot_cible, mot_contexte)
inputs, labels = tf.keras.preprocessing.sequence.skipgrams(
    sequence=ids_mots_exemple,       #Séquence d'IDs de mots (exemple ici)
    vocabulary_size=taille_vocabulaire,  #Taille totale du vocabulaire
    window_size=taille_fenetre,      #Nombre de mots à gauche et droite du mot cible à considérer
    negative_samples=1.0,            #Ratio de mots négatifs générés par mot positif (1 = autant de négatifs que de positifs)
    shuffle=False,                   #Ne pas mélanger les paires générées
    categorical=False,               #Sortie sous forme d'entiers (et non one-hot)
    sampling_table=None,             #Table d'échantillonnage négative (None = utilisation par défaut)
    seed=None                       #Graine aléatoire pour la reproductibilité (None = aléatoire)
)


In [ ]:
print("Exemple de paires skip-gram générées :")

for mot_entree, mot_label in zip(inputs, labels):
    try:
        mots_entree = [tokeniseur.index_word[wi] for wi in mot_entree]
        mot_label_str = tokeniseur.index_word[mot_label]
        print(f"\tEntrée : {mot_entree} ({mots_entree}) / Label : {mot_label}")
    except KeyError:
        #Si le mot n’est pas dans le vocabulaire (ex : ID = 0), on ignore
        continue


Exemple de paires skip-gram générées :
	Entrée : [130, 108] (['wales', 'coach']) / Label : 1
	Entrée : [108, 130] (['coach', 'wales']) / Label : 1
	Entrée : [108, 8848] (['coach', 'elated']) / Label : 1
	Entrée : [8848, 108] (['elated', 'coach']) / Label : 1
	Entrée : [8848, 15] (['elated', 'with']) / Label : 1
	Entrée : [15, 8848] (['with', 'elated']) / Label : 1
	Entrée : [15, 54] (['with', 'win']) / Label : 1
	Entrée : [54, 15] (['win', 'with']) / Label : 1


#Generation des candidats négative





In [ ]:
#Génération des paires (mot_cible, mot_contexte) en mode Skip-Gram
#Ici, on ne génère que des exemples positifs (negative_samples=0)
inputs, labels = tf.keras.preprocessing.sequence.skipgrams(
    ids_mots_exemple,                  #séquence de mots (sous forme d'IDs) à traiter
    vocabulary_size=len(tokeniseur.word_index.items()) + 1,  #Taille totale du vocabulaire (+1 pour le padding/index 0)
    window_size=taille_fenetre,              #Taille de la fenêtre contextuelle autour du mot cible
    negative_samples=0,                      #Pas d'exemples négatifs générés
    shuffle=False                            #Ne pas mélanger les paires générées
)

#Conversion des listes Python en tableaux NumPy, utile pour l'entraînement
inputs = np.array(inputs)
labels = np.array(labels)


In [ ]:
#Utilisation du sampling négatif pour générer des "faux" mots contexte
#à partir d'une vraie paire (mot_cible, mot_contexte)
taille_vocabulaire=15000+1
negative_sampling_candidates, true_expected_count, sampled_expected_count = tf.random.log_uniform_candidate_sampler(

    true_classes=inputs[:1, 1:],   #Les "vrais" mots contexte pour le premier exemple (doit être de forme [batch_size, num_true])
    num_true=1,                    #Nombre de mots positifs par exemple (ici : 1 mot contexte réel)
    num_sampled=10,                #Nombre d'exemples négatifs à générer
    unique=True,                   #Empêche les doublons parmi les exemples négatifs
    range_max=taille_vocabulaire,             #Intervalle dans lequel chercher les ID négatifs (0 à vocab_size - 1)
    name="negative_sampling"       #Nom de l'opération (utile dans un graphe TensorFlow)
)


In [ ]:
#Affichage de l'exemple positif (le mot contexte réel pour le premier mot cible)
print(f"Exemple positif (vrai mot contexte) : {inputs[:1, 1:]}")

#Affichage des exemples négatifs générés par TensorFlow
print(f"Exemples négatifs générés : {negative_sampling_candidates}")

#Fréquence attendue (statistique théorique) du mot contexte réel dans le vocabulaire
print(f"Fréquence attendue du vrai mot contexte : {true_expected_count}")

#Fréquence attendue des exemples négatifs (tirés de la distribution uniforme logarithmique)
print(f"Fréquence attendue des mots négatifs : {sampled_expected_count}")


Exemple positif (vrai mot contexte) : [[108]]
Exemples négatifs générés : [3522  872 6363   93  126 3529  438    5    4  305]
Fréquence attendue du vrai mot contexte : [[0.00949724]]
Fréquence attendue des mots négatifs : [2.9514404e-04 1.1905440e-03 1.6339698e-04 1.1004759e-02 8.1564346e-03
 2.9455888e-04 2.3661901e-03 1.6030747e-01 1.8960349e-01 3.3929560e-03]


In [ ]:
#Création d'une table d'échantillonnage pour la technique de "subsampling"
#Elle permet de réduire la fréquence des mots très courants (comme "le", "et", etc.)
sampling_table = tf.keras.preprocessing.sequence.make_sampling_table(
    size=taille_vocabulaire,           #Taille du vocabulaire (nombre total de mots différents)
    sampling_factor=1e-5    #Facteur de réduction : plus il est petit, plus les mots fréquents seront ignorés
)

#Affichage de la table d'échantillonnage générée
print("Table d'échantillonnage générée :")
print(sampling_table)


Table d'échantillonnage générée :
[0.00315225 0.00315225 0.00547597 ... 1.         1.         1.        ]



    Générateur de données Skip-Gram avec échantillonnage négatif
    pour l'entraînement d'un modèle Word2Vec.
    
    Paramètres :
    sequences : liste de séquences de mots (IDs)
    window_size : taille de la fenêtre contextuelle
    batch_size : nombre d'exemples par lot
    negative_samples : nombre de mots négatifs à générer par exemple
    vocab_size : taille du vocabulaire
    seed : graine aléatoire (optionnelle, pour reproductibilité)


In [ ]:
def skip_gram_data_generator(sequences, window_size, batch_size, negative_samples, vocab_size, seed=None):

    # Mélanger les indices des séquences
    rand_sequence_ids = np.arange(len(sequences))
    np.random.shuffle(rand_sequence_ids)

    # Parcourir chaque séquence de manière aléatoire
    for si in rand_sequence_ids:

        # Génération des paires Skip-Gram positives (target, context) pour une séquence
        positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
            sequences[si],
            vocabulary_size=vocab_size,
            window_size=window_size,
            negative_samples=0.0,   # Ici on ne génère que les paires positives
            shuffle=False,
            sampling_table=sampling_table,  # Réduction des mots fréquents
            seed=seed
        )

        targets, contexts, labels = [], [], []

        # Pour chaque paire positive (target_word, context_word)
        for target_word, context_word in positive_skip_grams:
            context_class = tf.expand_dims(tf.constant([context_word], dtype="int64"), 1)

            # Génération des exemples négatifs (ID de mots qui ne sont pas le contexte réel)
            negative_sampling_candidates, _, _ = tf.random.log_uniform_candidate_sampler(
                true_classes=context_class,
                num_true=1,
                num_sampled=negative_samples,
                unique=True,
                range_max=vocab_size,
                name="negative_sampling"
            )

            # Construction du vecteur contextuel (contexte réel + négatifs)
            context = tf.concat(
                [tf.constant([context_word], dtype='int64'), negative_sampling_candidates],
                axis=0
            )

            # Étiquettes : 1 pour vrai contexte, 0 pour les contextes négatifs
            label = tf.constant([1] + [0] * negative_samples, dtype="int64")

            # Ajout dans les listes globales
            targets.extend([target_word] * (negative_samples + 1))
            contexts.append(context)
            labels.append(label)

        # Conversion des listes en arrays pour traitement vectoriel
        contexts = np.concatenate(contexts)
        targets = np.array(targets)
        labels = np.concatenate(labels)

        # Vérification des dimensions
        assert contexts.shape[0] == targets.shape[0]
        assert contexts.shape[0] == labels.shape[0]

        # Graine aléatoire si non définie
        if not seed:
            seed = random.randint(0, int(10e6))

        # Mélange synchrone des données (même ordre)
        np.random.seed(seed)
        np.random.shuffle(contexts)
        np.random.seed(seed)
        np.random.shuffle(targets)
        np.random.seed(seed)
        np.random.shuffle(labels)

        # Génération des mini-lots de données
        for eg_id_start in range(0, contexts.shape[0], batch_size):
            yield (
                targets[eg_id_start: min(eg_id_start + batch_size, targets.shape[0])],
                contexts[eg_id_start: min(eg_id_start + batch_size, contexts.shape[0])]
            ), labels[eg_id_start: min(eg_id_start + batch_size, labels.shape[0])]


 Création du générateur de données Skip-Gram
 Paramètres :
 - news_sequences : les textes transformés en séquences de mots (IDs)
 - 4 : taille de la fenêtre contextuelle autour du mot cible
 - 10 : taille du batch (nombre d'exemples par lot)
 - 5 : nombre de mots négatifs à générer par exemple positif
 - n_vocab : taille du vocabulaire

In [ ]:
news_skip_gram_gen = skip_gram_data_generator(
    sequences_articles,       # Séquences de texte sous forme d’IDs
    4,                    # Taille de la fenêtre contextuelle
    10,                   # Batch size
    5,                    # Nombre d’échantillons négatifs
    taille_vocabulaire               # Taille du vocabulaire
)

# Boucle sur le générateur pour afficher un premier lot (batch)
for btc, bl in news_skip_gram_gen:
    # btc est un tuple : (target_words, context_words)
    print("Batch de mots cibles et contextes :")
    print(btc)

    # bl est un tableau d’étiquettes : 1 pour vrai contexte, 0 pour négatifs
    print("Étiquettes associées :")
    print(bl)

    # On affiche seulement le premier lot, donc on interrompt la boucle
    break


Batch de mots cibles et contextes :
(array([ 5611,  2363, 13806,   309,    42, 13808,  1342,  1775,  3899,
        5611]), array([   1, 3885,    0,    0,   13,  107, 2854,  532,   73,  175]))
Étiquettes associées :
[0 0 0 0 0 0 0 0 0 0]


# Implémentation de l'architecture skip-gram avec TensorFlow

Définition des hyperparamètres

In [ ]:
batch_size = 4096            #Nombre d'exemples dans un seul lot (batch)
embedding_size = 128         #Dimension du vecteur d'embedding (taille du vecteur de mots)
window_size = 1              #Taille de la fenêtre contextuelle : 1 mot de chaque côté du mot cible
negative_samples = 4         #Nombre d'échantillons négatifs générés pour chaque exemple positif
epochs = 5                   #Nombre d'époques (passes complètes sur les données) pour l'entraînement

#On choisit un ensemble de validation aléatoire pour évaluer la similarité des mots voisins
valid_size = 16              #Nombre de mots dans l'ensemble de validation
valid_window = 250           #Fenêtre large dans laquelle on choisit aléatoirement les mots pour validation

#Pour la reproductibilité, on fixe les seeds des générateurs aléatoires
np.random.seed(54321)
random.seed(54321)

#Sélection aléatoire de mots fréquents dans la fenêtre [0, valid_window)
valid_term_ids = np.array(random.sample(range(valid_window), valid_size))

#Ajout de mots moins fréquents issus de la fenêtre [1000, 1000+valid_window)
valid_term_ids = np.append(
    valid_term_ids,
    random.sample(range(1000, 1000 + valid_window), valid_size),
    axis=0
)


Défintion de modèle

In [ ]:
import tensorflow.keras.backend as K

# Nettoyage de la session Keras (pour repartir proprement)
K.clear_session()

# Définition des entrées du modèle : mot cible et mot contexte (des scalaires)
input_target = tf.keras.layers.Input(shape=(), name='target')
input_context = tf.keras.layers.Input(shape=(), name='context')

# Couche d'embedding pour le mot cible
target_embedding_layer = tf.keras.layers.Embedding(
    input_dim=taille_vocabulaire,        # taille du vocabulaire
    output_dim=embedding_size, # dimension des vecteurs embeddings
    name='target_embedding'
)

# Couche d'embedding pour le mot contexte
context_embedding_layer = tf.keras.layers.Embedding(
    input_dim=taille_vocabulaire,
    output_dim=embedding_size,
    name='context_embedding'
)

# Application des embeddings aux entrées
target_out = target_embedding_layer(input_target)   # vecteur embedding du mot cible
context_out = context_embedding_layer(input_context) # vecteur embedding du mot contexte

# Calcul du produit scalaire entre les embeddings cible et contexte
dot_product = tf.keras.layers.Dot(axes=-1)([context_out, target_out])

# Définition du modèle Keras avec deux entrées et un output (score de similarité)
skip_gram_model = tf.keras.models.Model(
    inputs=[input_target, input_context],
    outputs=dot_product,
    name='skip_gram_model'
)

# Compilation du modèle avec la fonction de perte BinaryCrossentropy (logits)
# car on fait de la classification binaire (mot contexte vrai ou faux)
skip_gram_model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer='adam',
    metrics=['accuracy']
)

# Affichage du résumé du modèle
skip_gram_model.summary()

Model: "skip_gram_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ context             │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ target (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context_embedding   │ (None, 128)       │  1,920,128 │ context[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ target_embedding    │ (None, 128)       │  1,920,128 │ target[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ context_embeddin… │
│                     │                   │            │ target_embedding… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,840,256 (14.65 MB)

 Trainable params: 3,840,256 (14.65 MB)

 Non-trainable params: 0 (0.00 B)

#Calcul les similaritées des mots

In [ ]:
import numpy as np

import tensorflow as tf



class ValidationCallback(tf.keras.callbacks.Callback):

    def __init__(self, valid_term_ids, model_with_embeddings, tokenizer):

        super().__init__()

        self.valid_term_ids = valid_term_ids                    # IDs des mots à valider

        self.model_with_embeddings = model_with_embeddings      # modèle avec couche d'embeddings

        self.tokenizer = tokenizer                              # tokenizer pour retrouver les mots



    def on_epoch_end(self, epoch, logs=None):

        """ Logique de validation à la fin de chaque époque """



        # Récupérer les poids de la couche d'embedding contexte

        embedding_weights = self.model_with_embeddings.get_layer("context_embedding").get_weights()[0]



        # Normaliser les vecteurs d'embeddings (norme L2)

        normalized_embeddings = embedding_weights / np.sqrt(np.sum(embedding_weights**2, axis=1, keepdims=True))



        # Extraire les embeddings des mots à valider

        valid_embeddings = normalized_embeddings[self.valid_term_ids, :]



        # Calculer la similarité (produit scalaire) entre mots validés et tout le vocabulaire

        similarity = np.dot(valid_embeddings, normalized_embeddings.T)



        top_k = 5  # On veut afficher les 5 mots les plus proches



        # Récupérer les indices des top_k mots les plus similaires (en excluant le mot lui-même)

        # Sort indices in descending order of similarity, then take top k+1 (including self)

        similarity_indices = np.argsort(-similarity, axis=1)[:, :top_k + 1]



        # Affichage des mots similaires

        for i, term_id in enumerate(self.valid_term_ids):

            # Get indices of similar words, excluding the word itself (which is the first one)

            similar_indices = similarity_indices[i, 1:]



            # Filter out indices that are not in the tokenizer's word_index (excluding 0 and others if not present)

            # Also filter out indices that map to the OOV token if it's defined and its index is > 1

            valid_similar_words = [

                self.tokenizer.index_word[j]

                for j in similar_indices

                if j in self.tokenizer.index_word and self.tokenizer.index_word[j] != self.tokenizer.oov_token

            ]



            # Join the valid similar words

            similar_words_str = ', '.join(valid_similar_words)



            print(f"{self.tokenizer.index_word[term_id]}: {similar_words_str}")

        print('\n')

#Exécuter l'Algorithme de Skip-Gram

In [ ]:
#Création du callback de validation pour suivre les embeddings à chaque époque
skipgram_validation_callback = ValidationCallback(valid_term_ids, skip_gram_model, tokeniseur)

for epoch in range(epochs):
    print(f"Début de l'époque {epoch + 1} / {epochs}")

    #Générateur de données Skip-gram pour cette époque
    news_skip_gram_gen = skip_gram_data_generator(
        sequences_articles,      #Séquences de texte sous forme d'IDs
        window_size,         #Taille de la fenêtre contextuelle
        batch_size,          #Taille des lots pour l'entraînement
        negative_samples,    #Nombre d'exemples négatifs par exemple positif
        taille_vocabulaire              #Taille du vocabulaire
    )

    #Entraînement du modèle pour une époque
    skip_gram_model.fit(
        news_skip_gram_gen,
        epochs=1,
        callbacks=[skipgram_validation_callback]
    )

Début de l'époque 1 / 5
    737/Unknown 75s 101ms/step - accuracy: 0.8000 - loss: 0.6823because: not, for, they
know: he, have, it
it: was, he, they
players: he, game, for
one: two, it, make
home: it, ball, game
international: ball, made, some
club: had, game, think
up: it, is, game
bbc: he, game, think
so: two, game, not
great: are, had, think
title: said, it, they
just: first, had, is
open: had, him, this
by: need, over, two
eighth: reassess, cruelly, apply, mainly, hand
thompson: separated, emotive, criticism, anoeta, colossus
met: moved, have, is, saturday, first
afterwards: player, will, team, some, is
plenty: we, second, he, have, right
ac: win, think, would
men: line, 2004, four, him
name: could, decided, bbc, far, off
contest: want, game, but, six, saturday
inzamam: hall, hurdles, performance, rifled, greek
throughout: france, game, out
cap: been, forced, play, an, ntini
gunners: run, great, care, football, already
rain: championship, played, if, reviewed, place
starts: but, is

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


    737/Unknown 69s 93ms/step - accuracy: 0.8000 - loss: 0.5486because: into, run, within
know: lead, think, get, want
it: know, think, going, lead, mistakes
players: go, there, get
one: couple, series, wish, slip, front
home: ball, 2004, chance, way
international: series, angel, coming, willing, front
club: chance, get, lot, time
up: want, thinking, won
bbc: games, hopefully, game, britain, leinster
so: lead, got, get
great: happen, lot, need
title: end, get, failing, 27, failed
just: lead, get, need
open: order, stay, irish, entertaining, final
by: sella, need, innings
eighth: crosses, there, difficult, relief, briton
thompson: players, criticism, forced, results, 15
met: decided, moved, go, get, you
afterwards: done, go, win, field, lot
plenty: lead, get, go
ac: play, score, win, end, saying
men: 2004, line, lot
name: happen, expected, 2004, seen, decided
contest: want, yards, once, doing, know
inzamam: hall, scored, managed, ensure, work
throughout: lot, ensure, need
cap: forced, n